In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import shutil

zip_path = "/content/drive/MyDrive/Colab Notebooks/DeepRetStroke/DL.zip"

shutil.copy(zip_path, "/content/")

'/content/DL.zip'

In [4]:
import zipfile

with zipfile.ZipFile("/content/DL.zip", "r") as zip_ref:
    zip_ref.extractall("/content/dataset")

print("Dataset Extracted Successfully!")

Dataset Extracted Successfully!


In [5]:
import os

print(os.listdir("/content/dataset"))

['ODIR-5K']


In [6]:
import os

print(os.listdir("/content/dataset/ODIR-5K"))

['training', 'data.xlsx', 'full_df.csv', 'testing']


In [7]:
import pandas as pd

df = pd.read_csv("/content/dataset/ODIR-5K/full_df.csv")

print(df.head())

   ID  Patient Age Patient Sex Left-Fundus Right-Fundus  \
0   0           69      Female  0_left.jpg  0_right.jpg   
1   1           57        Male  1_left.jpg  1_right.jpg   
2   2           42        Male  2_left.jpg  2_right.jpg   
3   4           53        Male  4_left.jpg  4_right.jpg   
4   5           50      Female  5_left.jpg  5_right.jpg   

                            Left-Diagnostic Keywords  \
0                                           cataract   
1                                      normal fundus   
2  laser spot，moderate non proliferative retinopathy   
3                        macular epiretinal membrane   
4             moderate non proliferative retinopathy   

                Right-Diagnostic Keywords  N  D  G  C  A  H  M  O labels  
0                           normal fundus  0  0  0  1  0  0  0  0  ['N']  
1                           normal fundus  1  0  0  0  0  0  0  0  ['N']  
2  moderate non proliferative retinopathy  0  1  0  0  0  0  0  1  ['D']  
3       

In [8]:
print(df.shape)
print(df.columns)

(6392, 16)
Index(['ID', 'Patient Age', 'Patient Sex', 'Left-Fundus', 'Right-Fundus',
       'Left-Diagnostic Keywords', 'Right-Diagnostic Keywords', 'N', 'D', 'G',
       'C', 'A', 'H', 'M', 'O', 'labels'],
      dtype='object')


In [9]:
print(df.isnull().sum())

ID                           0
Patient Age                  0
Patient Sex                  0
Left-Fundus                  0
Right-Fundus                 0
Left-Diagnostic Keywords     0
Right-Diagnostic Keywords    0
N                            0
D                            0
G                            0
C                            0
A                            0
H                            0
M                            0
O                            0
labels                       0
dtype: int64


In [10]:
import pandas as pd

label_cols = ['N','D','G','C','A','H','M','O']

new_data = []

for _, row in df.iterrows():

    left_image = row['Left-Fundus']

    new_data.append({
        'filename': left_image,
        'N': row['N'],
        'D': row['D'],
        'G': row['G'],
        'C': row['C'],
        'A': row['A'],
        'H': row['H'],
        'M': row['M'],
        'O': row['O']
    })

    right_image = row['Right-Fundus']

    new_data.append({
        'filename': right_image,
        'N': row['N'],
        'D': row['D'],
        'G': row['G'],
        'C': row['C'],
        'A': row['A'],
        'H': row['H'],
        'M': row['M'],
        'O': row['O']
    })

final_df = pd.DataFrame(new_data)

print(final_df.head())
print()
print("Total Images :", len(final_df))

      filename  N  D  G  C  A  H  M  O
0   0_left.jpg  0  0  0  1  0  0  0  0
1  0_right.jpg  0  0  0  1  0  0  0  0
2   1_left.jpg  1  0  0  0  0  0  0  0
3  1_right.jpg  1  0  0  0  0  0  0  0
4   2_left.jpg  0  1  0  0  0  0  0  1

Total Images : 12784


In [11]:
final_df.to_csv("/content/final_dataset.csv", index=False)

print("CSV Saved Successfully!")

CSV Saved Successfully!


In [12]:
df_final = pd.read_csv("/content/final_dataset.csv")

print(df_final.head())
print(df_final.shape)

      filename  N  D  G  C  A  H  M  O
0   0_left.jpg  0  0  0  1  0  0  0  0
1  0_right.jpg  0  0  0  1  0  0  0  0
2   1_left.jpg  1  0  0  0  0  0  0  0
3  1_right.jpg  1  0  0  0  0  0  0  0
4   2_left.jpg  0  1  0  0  0  0  0  1
(12784, 9)


In [13]:
import shutil

shutil.copy(
    "/content/final_dataset.csv",
    "/content/drive/MyDrive/Colab Notebooks/DeepRetStroke/final_dataset.csv"
)

print("Saved to Google Drive!")

Saved to Google Drive!


In [14]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    final_df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Training Images :", len(train_df))
print("Validation Images :", len(val_df))

Training Images : 10227
Validation Images : 2557


In [15]:
train_df.to_csv("/content/train.csv", index=False)
val_df.to_csv("/content/validation.csv", index=False)

print("Train & Validation CSV Saved Successfully!")

Train & Validation CSV Saved Successfully!


In [16]:
import shutil

shutil.copy("/content/train.csv",
            "/content/drive/MyDrive/Colab Notebooks/DeepRetStroke/train.csv")

shutil.copy("/content/validation.csv",
            "/content/drive/MyDrive/Colab Notebooks/DeepRetStroke/validation.csv")

print("Files Saved to Google Drive!")

Files Saved to Google Drive!


In [17]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.20.0


In [18]:
train_df = pd.read_csv("/content/train.csv")
val_df = pd.read_csv("/content/validation.csv")

print(train_df.head())

         filename  N  D  G  C  A  H  M  O
0   4080_left.jpg  0  1  0  0  0  0  0  0
1     64_left.jpg  0  1  0  0  0  0  0  1
2  2873_right.jpg  1  0  0  0  0  0  0  0
3  2419_right.jpg  1  0  0  0  0  0  0  0
4    862_left.jpg  0  0  0  0  0  0  0  1


In [19]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

LABELS = ['N','D','G','C','A','H','M','O']

In [20]:
def load_image(filename, labels):

    image_path = tf.strings.join([
        "/content/dataset/ODIR-5K/training/",
        filename
    ])

    image = tf.io.read_file(image_path)

    image = tf.image.decode_jpeg(image, channels=3)

    image = tf.image.resize(image, IMG_SIZE)

    image = image / 255.0

    return image, labels

In [21]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        train_df["filename"].values,
        train_df[LABELS].values.astype("float32")
    )
)

train_dataset = train_dataset.map(load_image)

train_dataset = train_dataset.shuffle(1000)

train_dataset = train_dataset.batch(BATCH_SIZE)

train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

In [22]:
val_dataset = tf.data.Dataset.from_tensor_slices(
    (
        val_df["filename"].values,
        val_df[LABELS].values.astype("float32")
    )
)

val_dataset = val_dataset.map(load_image)

val_dataset = val_dataset.batch(BATCH_SIZE)

val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)

print("TensorFlow Dataset Created Successfully!")

TensorFlow Dataset Created Successfully!


In [23]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.1),
])

In [24]:
base_model = keras.applications.EfficientNetB3(
    include_top=False,
    weights="imagenet",
    input_shape=(224,224,3)
)

base_model.trainable = True

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [25]:
inputs = keras.Input(shape=(224,224,3))

x = data_augmentation(inputs)

x = keras.applications.efficientnet.preprocess_input(x)

x = base_model(x, training=True)

x = layers.GlobalAveragePooling2D()(x)

x = layers.Dropout(0.4)(x)

x = layers.Dense(256, activation="relu")(x)

x = layers.Dropout(0.4)(x)

outputs = layers.Dense(8, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb3 (Functional)     │ (None, 7, 7, 1536)     │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       393,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         2,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,179,063 (42.64 MB)

 Trainable params: 11,091,760 (42.31 MB)

 Non-trainable params: 87,303 (341.03 KB)

In [26]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),

    loss="binary_crossentropy",

    metrics=[
        keras.metrics.BinaryAccuracy(name="bin_acc"),
        keras.metrics.AUC(name="roc_auc", multi_label=True)
    ]
)

print("Model Compiled Successfully!")

Model Compiled Successfully!


In [27]:
from tensorflow.keras import callbacks

early_stop = callbacks.EarlyStopping(
    monitor="val_roc_auc",
    patience=12,
    mode="max",
    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.7,
    patience=4,
    min_lr=1e-7
)

checkpoint = callbacks.ModelCheckpoint(
    "/content/drive/MyDrive/Colab Notebooks/DeepRetStroke/efficientnetb3_best.keras",

    monitor="val_roc_auc",

    mode="max",

    save_best_only=True,

    verbose=1
)

print("Callbacks Ready!")

Callbacks Ready!


In [ ]:
EPOCHS = 60

print("🚀 Training EfficientNetB3...")

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=[
        checkpoint,
        reduce_lr,
        early_stop
    ],
    verbose=1
)

🚀 Training EfficientNetB3...
Epoch 1/60
320/320 ━━━━━━━━━━━━━━━━━━━━ 0s 687ms/step - bin_acc: 0.6939 - loss: 0.5872 - roc_auc: 0.5057
Epoch 1: val_roc_auc improved from None to 0.56823, saving model to /content/drive/MyDrive/Colab Notebooks/DeepRetStroke/efficientnetb3_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Colab Notebooks/DeepRetStroke/efficientnetb3_best.keras
320/320 ━━━━━━━━━━━━━━━━━━━━ 343s 811ms/step - bin_acc: 0.7691 - loss: 0.5087 - roc_auc: 0.5075 - val_bin_acc: 0.8315 - val_loss: 0.4590 - val_roc_auc: 0.5682 - learning_rate: 1.0000e-05
Epoch 2/60
320/320 ━━━━━━━━━━━━━━━━━━━━ 0s 698ms/step - bin_acc: 0.8383 - loss: 0.3994 - roc_auc: 0.5182
Epoch 2: val_roc_auc did not improve from 0.56823
320/320 ━━━━━━━━━━━━━━━━━━━━ 266s 792ms/step - bin_acc: 0.8403 - loss: 0.3913 - roc_auc: 0.5209 - val_bin_acc: 0.8540 - val_loss: 0.3856 - val_roc_auc: 0.5523 - learning_rate: 1.0000e-05
Epoch 3/60
320/320 ━━━━━━━━━━━━━━━━━━━━ 0s 700ms/step - bin_acc: 0.8416 - lo